# 01 — Title/abstract screening

Screen a project's title/abstract queue. Every keystroke appends to
`decisions.jsonl` immediately, so closing this notebook — or losing the kernel —
costs at most the record on screen.

Records are presented in an order seeded from the project slug and uncorrelated
with citation count or any other ranking. That is deliberate: a ranked queue
imports the ranker's judgement into a protocol that is supposed to be yours.

Author names and citation counts are hidden by default for the same reason. Pass
`blind=False` if your protocol genuinely needs them.

In [ ]:
import os

import panel as pn

import prismabib
from prismabib.errors import ConfigError
from prismabib.project import Project

print(f"prismabib {prismabib.__version__}")

# Set PRISMABIB_NOTEBOOK_SLUG to screen your own project. The default is the
# bundled reference fixture, so this notebook executes in CI without a Scopus
# key and without touching anyone's real review.
SLUG = os.environ.get("PRISMABIB_NOTEBOOK_SLUG", "reference")
REVIEWER = os.environ.get("PRISMABIB_NOTEBOOK_REVIEWER", "reviewer")

# Caught rather than raised. Under `panel serve`, an exception here kills the
# whole document and the browser gets a valid, empty page -- the traceback goes
# to the terminal, which someone who launched with `--show` is not looking at.
# A blank screen is the least debuggable failure a UI has, so the error is
# carried to the last cell and rendered where the person actually is.
project = None
STARTUP_ERROR = None
try:
    project = Project.open(SLUG)
    print(f"screening {project.slug} at {project.root}")
except ConfigError as error:
    STARTUP_ERROR = str(error)
    print(STARTUP_ERROR)

## Build the store if it is not there yet

Layer 1 is derived, never authored: it is rebuilt from the immutable Layer 0
archive by one function call, so deleting it costs nothing but time.

In [ ]:
from prismabib.store.load import build_store

if project is not None and not project.db_path.exists():
    stats = build_store(project, rebuild=True)
    print(f"built: {stats.records_loaded} records")

## Where you are

Run this before screening and again whenever you want to see progress. It reads
the decision log, so it stays accurate across restarts.

In [ ]:
from prismabib.screening.queue import screening_queue
from prismabib.stage import PrismaStage

if project is not None:
    queue = screening_queue(project, PrismaStage.TITLE_ABSTRACT, REVIEWER)
    print(f"{queue.decided} decided / {queue.total} total — {queue.remaining} remaining")

## Screen

`i` include · `e` then a digit exclude · `u` unsure · `n`/`p` move · `z` undo · `?` help

`unsure` never resolves to inclusion — the record stays in the queue for a
second pass and is reported separately in the PRISMA diagram, rather than being
folded into your exclusions.

In [ ]:
from prismabib.screening.ui import screener

# A conditional *expression*, so this stays the cell's last expression and Panel
# renders it either way. An `if`/`else` block would display nothing at all on
# the error branch, which is the failure this exists to remove.
(
    screener(project, stage="title_abstract", reviewer=REVIEWER)
    if project is not None
    else pn.pane.Markdown(
        "## Screening cannot start\n\n"
        f"```\n{STARTUP_ERROR}\n```\n\n"
        "This notebook defaults to the bundled `reference` fixture. To screen "
        "your own project, set the slug before launching:\n\n"
        "```bash\n"
        "PRISMABIB_NOTEBOOK_SLUG=<your-slug> PRISMABIB_NOTEBOOK_REVIEWER=<your-name> \\\n"
        "  uv run panel serve notebooks/01_screen_title_abstract.ipynb --show\n"
        "```\n\n"
        "`prismabib` lists nothing, so the slug is the directory name under your "
        "projects root.",
        styles={"color": "#8a1c1c"},
    )
)

## Then

`prismabib flow <slug>` prints the PRISMA counts, drawn from the decision log —
no number in it is typed by hand.